first, we load our youtube csv dataset and look through all records and go through them

In [ ]:
import pandas as pd

df = pd.read_csv("youtube_comments.csv")

print(df.shape)
df.head()

(29050, 6)


,author,updated_at,like_count,text,video_id,public
0,@doomkid1331,2026-05-23T23:42:49Z,0,"Wore my Mk4s until they broke, and exchanged t...",XbXXKhfvSF0,True
1,@TurkeyTomDeezNuts,2026-05-23T16:48:27Z,0,These have such bad reviews,XbXXKhfvSF0,True
2,@elevate5136,2026-05-22T04:30:00Z,0,I bought these in South Africa for my 12 hour ...,XbXXKhfvSF0,True
3,@Tdizzle_91,2026-05-21T20:57:05Z,0,Can you compare these to the new collextion mo...,XbXXKhfvSF0,True
4,@andrewbarf,2026-05-21T09:35:58Z,0,Excellent review Marques; all the main points ...,XbXXKhfvSF0,True


In [ ]:
df = df.dropna(subset=["text"]) # this is to remove null values
df = df.drop_duplicates(subset=["text"]) # to remove any duplicates
print(df.shape)

(27738, 6)


In [ ]:
df["clean_text"] = df["text"] #create another column to store clean text
df["clean_text"] = df["clean_text"].str.lower()  #converts to lowercase

import re

def remove_urls(text):
    return re.sub(r"http\S+|www\S+", "", str(text))

df["clean_text"] = df["clean_text"].apply(remove_urls)

In [ ]:
def remove_emojis(text):
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "]+",
        flags=re.UNICODE,
    )
    return emoji_pattern.sub("", text)

df["clean_text"] = df["clean_text"].apply(remove_emojis)

In [ ]:
def remove_special_chars(text):
    return re.sub(r"[^a-zA-Z0-9\s]", "", text)

df["clean_text"] = df["clean_text"].apply(remove_special_chars)

In [ ]:
df["clean_text"] = df["clean_text"].str.replace(r"\s+", " ", regex=True) #removes any extra spaces
df["clean_text"] = df["clean_text"].str.strip()
df = df[df["clean_text"].str.split().str.len() >= 3]   #removes very short comments and keeps min 3 words
df.head()

,author,updated_at,like_count,text,video_id,public,clean_text
0,@doomkid1331,2026-05-23T23:42:49Z,0,"Wore my Mk4s until they broke, and exchanged t...",XbXXKhfvSF0,True,wore my mk4s until they broke and exchanged th...
1,@TurkeyTomDeezNuts,2026-05-23T16:48:27Z,0,These have such bad reviews,XbXXKhfvSF0,True,these have such bad reviews
2,@elevate5136,2026-05-22T04:30:00Z,0,I bought these in South Africa for my 12 hour ...,XbXXKhfvSF0,True,i bought these in south africa for my 12 hour ...
3,@Tdizzle_91,2026-05-21T20:57:05Z,0,Can you compare these to the new collextion mo...,XbXXKhfvSF0,True,can you compare these to the new collextion mo...
4,@andrewbarf,2026-05-21T09:35:58Z,0,Excellent review Marques; all the main points ...,XbXXKhfvSF0,True,excellent review marques all the main points c...


In [ ]:
import re

def remove_timestamps(text):
    return re.sub(r'\b\d{1,2}:\d{2}(?::\d{2})?\b', '', str(text))

df["clean_text"] = df["clean_text"].apply(remove_timestamps)

In [ ]:
df["clean_text"] = df["clean_text"].str.replace(r"\s+", " ", regex=True)
df["clean_text"] = df["clean_text"].str.strip()

In [ ]:
print(df.shape)
print("clean_text" in df.columns)

(25847, 7)
True


In [ ]:
# ==========================================
# DOMAIN-AWARE SPELL CORRECTION
# ==========================================

import re

SPELLING_FIXES = {
    "reccomend": "recommend",
    "recomend": "recommend",
    "definately": "definitely",
    "definetely": "definitely",
    "seperately": "separately",
    "becuase": "because",
    "becouse": "because",
    "wierd": "weird",
    "alot": "a lot",
    "awsome": "awesome",
    "amazng": "amazing",
    "genuinly": "genuinely",
    "litterally": "literally",
    "realy": "really",
    "thier": "their",
    "wich": "which",
    "teh": "the",
    "woud": "would",
    "coud": "could",
    "dont": "don't",
    "doesnt": "doesn't",
    "wouldnt": "wouldn't",
    "cant": "can't",
    "im": "i'm",
    "garmiin": "garmin",
    "ipone": "iphone",
    "iphon": "iphone",
    "samsng": "samsung",
    "sonny": "sony",
    "batery": "battery",
    "comfot": "comfort",
    "relability": "reliability"
}

def conservative_spell_correct(text):
    corrected_text = str(text).lower()

    for incorrect, correct in SPELLING_FIXES.items():
        corrected_text = re.sub(
            rf"\b{re.escape(incorrect)}\b",
            correct,
            corrected_text
        )

    return corrected_text


# Keep clean_text unchanged so existing results stay reproducible
df["spell_corrected_text"] = df["clean_text"].apply(
    conservative_spell_correct
)

changed_examples = df.loc[
    df["clean_text"] != df["spell_corrected_text"],
    ["clean_text", "spell_corrected_text"]
].head(10)

print("Comments changed by spell correction:",
      (df["clean_text"] != df["spell_corrected_text"]).sum())

display(changed_examples)

Comments changed by spell correction: 3184


,clean_text,spell_corrected_text
6,i dont like the design at all but they are app...,i don't like the design at all but they are ap...
12,still with the mx4 dont need upgrade,still with the mx4 don't need upgrade
13,i dont recommend them the sound quality isnt g...,i don't recommend them the sound quality isnt ...
20,had my beat solo pros since 2021 and they brok...,had my beat solo pros since 2021 and they brok...
25,still ridiculous in 2026 that whatever mics th...,still ridiculous in 2026 that whatever mics th...
31,i dont like bass heavy headphones,i don't like bass heavy headphones
33,i just bought these in greysilver they are ins...,i just bought these in greysilver they are ins...
34,i love the lack of watermoisture resistance ca...,i love the lack of watermoisture resistance ca...
35,i dont understand these reviews about the nois...,i don't understand these reviews about the noi...
39,march 2026 checking it im an apple ecosystem g...,march 2026 checking it i'm an apple ecosystem ...


In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 88.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
from tqdm import tqdm
import spacy

nlp = spacy.load("en_core_web_sm")

tqdm.pandas()

def tokenize(text):
    doc = nlp(text)

    tokens = []
    for token in doc:
        if not token.is_punct:
            tokens.append(token.text)

    return tokens

df["tokens"] = df["clean_text"].progress_apply(tokenize)

100%|██████████| 24623/24623 [03:23<00:00, 120.80it/s]


In [ ]:
def lemmatize(text):
    doc = nlp(text)

    lemmas = []
    for token in doc:
        if not token.is_stop and not token.is_punct:
            lemmas.append(token.lemma_)

    return lemmas

df["lemmas"] = df["clean_text"].progress_apply(lemmatize)

100%|██████████| 24623/24623 [03:14<00:00, 126.73it/s]


In [ ]:
df["processed_text"] = df["lemmas"].apply(lambda x: " ".join(x))

In [ ]:
print(df.columns)

Index(['author', 'updated_at', 'like_count', 'text', 'video_id', 'public',
       'clean_text', 'tokens', 'lemmas', 'processed_text'],
      dtype='object')


In [ ]:
df.head()

,author,updated_at,like_count,text,video_id,public,clean_text,tokens,lemmas,processed_text
0,@doomkid1331,2026-05-23T23:42:49Z,0,"Wore my Mk4s until they broke, and exchanged t...",XbXXKhfvSF0,True,wore my mk4s until they broke and exchanged th...,"[wore, my, mk4s, until, they, broke, and, exch...","[wear, mk4s, break, exchange, mk5, perfect, mk6]",wear mk4s break exchange mk5 perfect mk6
1,@TurkeyTomDeezNuts,2026-05-23T16:48:27Z,0,These have such bad reviews,XbXXKhfvSF0,True,these have such bad reviews,"[these, have, such, bad, reviews]","[bad, review]",bad review
2,@elevate5136,2026-05-22T04:30:00Z,0,I bought these in South Africa for my 12 hour ...,XbXXKhfvSF0,True,i bought these in south africa for my 12 hour ...,"[i, bought, these, in, south, africa, for, my,...","[buy, south, africa, 12, hour, flight, home, a...",buy south africa 12 hour flight home absolutel...
3,@Tdizzle_91,2026-05-21T20:57:05Z,0,Can you compare these to the new collextion mo...,XbXXKhfvSF0,True,can you compare these to the new collextion mo...,"[can, you, compare, these, to, the, new, colle...","[compare, new, collextion, model, know, buy, lol]",compare new collextion model know buy lol
4,@andrewbarf,2026-05-21T09:35:58Z,0,Excellent review Marques; all the main points ...,XbXXKhfvSF0,True,excellent review marques all the main points c...,"[excellent, review, marques, all, the, main, p...","[excellent, review, marque, main, point, cover...",excellent review marque main point cover waffl...


In [ ]:
df.to_csv("youtube_comments_preprocessedfinal.csv", index=False)

In [ ]:
print(df.shape)
df.head()

(24623, 10)


,author,updated_at,like_count,text,video_id,public,clean_text,tokens,lemmas,processed_text
0,@doomkid1331,2026-05-23T23:42:49Z,0,"Wore my Mk4s until they broke, and exchanged t...",XbXXKhfvSF0,True,wore my mk4s until they broke and exchanged th...,"[wore, my, mk4s, until, they, broke, and, exch...","[wear, mk4s, break, exchange, mk5, perfect, mk6]",wear mk4s break exchange mk5 perfect mk6
1,@TurkeyTomDeezNuts,2026-05-23T16:48:27Z,0,These have such bad reviews,XbXXKhfvSF0,True,these have such bad reviews,"[these, have, such, bad, reviews]","[bad, review]",bad review
2,@elevate5136,2026-05-22T04:30:00Z,0,I bought these in South Africa for my 12 hour ...,XbXXKhfvSF0,True,i bought these in south africa for my 12 hour ...,"[i, bought, these, in, south, africa, for, my,...","[buy, south, africa, 12, hour, flight, home, a...",buy south africa 12 hour flight home absolutel...
3,@Tdizzle_91,2026-05-21T20:57:05Z,0,Can you compare these to the new collextion mo...,XbXXKhfvSF0,True,can you compare these to the new collextion mo...,"[can, you, compare, these, to, the, new, colle...","[compare, new, collextion, model, know, buy, lol]",compare new collextion model know buy lol
4,@andrewbarf,2026-05-21T09:35:58Z,0,Excellent review Marques; all the main points ...,XbXXKhfvSF0,True,excellent review marques all the main points c...,"[excellent, review, marques, all, the, main, p...","[excellent, review, marque, main, point, cover...",excellent review marque main point cover waffl...


In [ ]:
from google.colab import files

files.download("youtube_comments_preprocessedfinal.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ==========================================
# DOMAIN-AWARE SPELL CORRECTION
# ==========================================

import re

SPELLING_FIXES = {
    "reccomend": "recommend",
    "recomend": "recommend",
    "definately": "definitely",
    "definetely": "definitely",
    "seperately": "separately",
    "becuase": "because",
    "becouse": "because",
    "wierd": "weird",
    "alot": "a lot",
    "awsome": "awesome",
    "amazng": "amazing",
    "genuinly": "genuinely",
    "litterally": "literally",
    "realy": "really",
    "thier": "their",
    "wich": "which",
    "teh": "the",
    "woud": "would",
    "coud": "could",
    "dont": "dont",
    "doesnt": "doesnt",
    "wouldnt": "wouldnt",
    "cant": "cant",
    "im": "im",
    "garmiin": "garmin",
    "ipone": "iphone",
    "iphon": "iphone",
    "samsng": "samsung",
    "sonny": "sony",
    "batery": "battery",
    "comfot": "comfort",
    "relability": "reliability"
}

def conservative_spell_correct(text):
    corrected_text = str(text).lower()

    for incorrect, correct in SPELLING_FIXES.items():
        corrected_text = re.sub(
            rf"\b{re.escape(incorrect)}\b",
            correct,
            corrected_text
        )

    return corrected_text


df["spell_corrected_text"] = df["clean_text"].apply(
    conservative_spell_correct
)

changed_examples = df.loc[
    df["clean_text"] != df["spell_corrected_text"],
    ["clean_text", "spell_corrected_text"]
].head(10)

print(
    "Comments changed:",
    (df["clean_text"] != df["spell_corrected_text"]).sum()
)

display(changed_examples)

Comments changed: 66


,clean_text,spell_corrected_text
242,i purchased xm5 earbuds and after 1 year its c...,i purchased xm5 earbuds and after 1 year its c...
364,hellllo guys i need help if your main thing is...,hellllo guys i need help if your main thing is...
873,im still rocking my m3 that i bought back in 2...,im still rocking my m3 that i bought back in 2...
1263,would they be cool for a dude who sweats alot,would they be cool for a dude who sweats a lot
1842,im still using my mx4 and they still work grea...,im still using my mx4 and they still work grea...
3622,new xm6s looks awsome,new xm6s looks awesome
4903,xm5 is trash mine broken after 15 years it jus...,xm5 is trash mine broken after 15 years it jus...
5833,cant believe that i disliked the video at such...,cant believe that i disliked the video at such...
5859,having huawei would change alot,having huawei would change a lot
5926,you should rename it to most expensive mid sea...,you should rename it to most expensive mid sea...


In [ ]:
df.to_csv(
    "youtube_comments_preprocessed_spellchecked.csv",
    index=False
)

print("Saved: youtube_comments_preprocessed_spellchecked.csv")

Saved: youtube_comments_preprocessed_spellchecked.csv


In [ ]:
from google.colab import files

files.download("youtube_comments_preprocessed_spellchecked.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>